In [ ]:
!pip install -q biopython

# Lecture 1: Strings and the Genome — SOLUTIONS

**Course:** Intro to Programming for Computational Biology  
**For instructors only — do not distribute to students.**

---
## Section 1: DNA as a Python String

In [ ]:
dna = "ATGGTGCATCTGACTCCTGAGGAGAAGTCT"
print("Sequence:", dna)
print("Length:", len(dna))

### Exercise 1.1 — 4th codon

In [ ]:
fourth_codon = dna[9:12]
print("4th codon:", fourth_codon)  # CTG

### Exercise 1.2 — GC content

In [ ]:
def gc_content(sequence):
    gc = sequence.count("G") + sequence.count("C")
    return gc / len(sequence)

print(gc_content(dna))  # ~0.567

### Exercise 1.3 — Split into codons

In [ ]:
codons = []
for i in range(0, len(dna), 3):
    codon = dna[i:i+3]
    codons.append(codon)

print(codons)
# ['ATG', 'GTG', 'CAT', 'CTG', 'ACT', 'CCT', 'GAG', 'GAG', 'AAG', 'TCT']

### Exercise 1.4 — ORF finding

In [ ]:
mystery = "TTAATGCGATAAATGGTGCATCTGACTCCTGAGGAGAAGTCTGAATAG"
stop_codons = ["TAA", "TAG", "TGA"]

# 1.4a — find all ATG positions
atg_positions = []
for i in range(len(mystery)):
    if mystery[i:i+3] == "ATG":
        atg_positions.append(i)

print("ATG positions:", atg_positions)  # [3, 12]

In [ ]:
# 1.4b — find each ORF
for start in atg_positions:
    for i in range(start, len(mystery), 3):
        codon = mystery[i:i+3]
        if codon in stop_codons:
            orf = mystery[start:i+3]
            print(f"ATG at {start}: ORF = {orf}  (length {len(orf)} bp)")
            break

# ATG at 3:  ORF = ATGCGATAA  (9 bp)  → MR   (2 aa) — short, likely not real
# ATG at 12: ORF = ATGGTGCATCTGACTCCTGAGGAGAAGTCTGAATAG  (36 bp) → MVHLTPEEKSE (11 aa) — correct ORF

---
## Section 2: From DNA to Protein

In [ ]:
mrna = dna.replace("T", "U")
print("mRNA:", mrna)

In [ ]:
codon_table = {
    "UUU": "F", "UUC": "F",
    "UUA": "L", "UUG": "L", "CUU": "L", "CUC": "L", "CUA": "L", "CUG": "L",
    "AUU": "I", "AUC": "I", "AUA": "I",
    "AUG": "M",
    "GUU": "V", "GUC": "V", "GUA": "V", "GUG": "V",
    "UCU": "S", "UCC": "S", "UCA": "S", "UCG": "S", "AGU": "S", "AGC": "S",
    "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "GCU": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "UAU": "Y", "UAC": "Y",
    "UAA": "*", "UAG": "*", "UGA": "*",
    "CAU": "H", "CAC": "H",
    "CAA": "Q", "CAG": "Q",
    "AAU": "N", "AAC": "N",
    "AAA": "K", "AAG": "K",
    "GAU": "D", "GAC": "D",
    "GAA": "E", "GAG": "E",
    "UGU": "C", "UGC": "C",
    "UGG": "W",
    "CGU": "R", "CGC": "R", "CGA": "R", "CGG": "R", "AGA": "R", "AGG": "R",
    "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}

### Exercise 2.1 — translate()

In [ ]:
def translate(dna_sequence):
    mrna = dna_sequence.replace("T", "U")
    protein = ""
    for i in range(0, len(mrna), 3):
        codon = mrna[i:i+3]
        amino_acid = codon_table.get(codon, "?")
        if amino_acid == "*":
            break
        protein += amino_acid
    return protein

print(translate(dna))  # MVHLTPEEKS

---
## Section 3: Introducing Biopython

In [ ]:
from Bio.Seq import Seq

seq = Seq(dna)
print("Complement:        ", seq.complement())
print("Reverse complement:", seq.reverse_complement())
print("Transcription:     ", seq.transcribe())
print("Translation:       ", seq.translate())

### Exercise 3.1 — Compare Biopython to manual

In [ ]:
bio_protein = str(seq.translate())
bio_protein_clean = bio_protein[:-1]  # remove trailing *

manual_result = translate(dna)
print("Biopython:", bio_protein_clean)
print("Manual:   ", manual_result)
print("Match:", bio_protein_clean == manual_result)

### 3.2 Full HBB CDS

In [ ]:
hbb_cds_str = (
    "ATGGTGCATCTGACTCCTGAGGAGAAGTCTGCCGTTACTGCCCTGTGGGGCAAGGTGAAC"
    "GTGGATGAAGTTGGTGGTGAGGCCCTGGGCAGGCTGCTGGTGGTCTACCCTTGGACCCAG"
    "AGGTTCTTTGAGTCCTTTGGGGATCTGTCCACTCCTGATGCTGTTATGGGCAACCCTAAG"
    "GTGAAGGCTCATGGCAAGAAAGTGCTCGGTGCCTTTAGTGATGGCCTGGCTCACCTGGAC"
    "AACCTCAAGGGCACCTTTGCCACACTGAGTGAGCTGCACTGTGACAAGCTGCACGTGGAT"
    "CCTGAGAACTTCAGGCTCCTGGGCAACGTGCTGGTCTGTGTGCTGGCCCATCACTTTGGC"
    "AAAGAATTCACCCCACCAGTGCAGGCTGCCTATCAGAAAGTGGTGGCTGGTGTGGCTAAT"
    "GCCCTGGCCCACAAGTATCACTAA"
)

hbb_cds = Seq(hbb_cds_str)
hbb_protein = hbb_cds.translate(to_stop=True)
print(hbb_protein)
print("Length:", len(hbb_protein), "amino acids")

### Exercise 3.2 — Glutamate count

In [ ]:
num_E = str(hbb_protein).count("E")
print("Glutamate (E) count:", num_E)
print("Amino acid at CDS position 7:", str(hbb_protein)[6])  # index 6 = codon 7 = mature position 6 = E

---
## Section 4: Mutations

In [ ]:
def point_mutation(sequence, position, new_base):
    return sequence[:position] + new_base + sequence[position + 1:]

# Verify codon 7 (positions 18-20)
print("CDS codon 7 (normal): ", hbb_cds_str[18:21])  # GAG

hbb_sickle = point_mutation(hbb_cds_str, 19, "T")
print("CDS codon 7 (sickle): ", hbb_sickle[18:21])   # GTG

In [ ]:
normal_protein = Seq(hbb_cds_str).translate(to_stop=True)
sickle_protein = Seq(hbb_sickle).translate(to_stop=True)

print("Normal amino acid at position 7: ", normal_protein[6])   # E
print("Sickle amino acid at position 7: ", sickle_protein[6])   # V
print("Proteins otherwise identical?    ", normal_protein[7:] == sickle_protein[7:])

### Exercise 4.1 — classify_mutation()

In [ ]:
def classify_mutation(original_cds, mutated_cds):
    original_protein = str(Seq(original_cds).translate())
    mutated_protein  = str(Seq(mutated_cds).translate())

    if "*" in mutated_protein and mutated_protein.index("*") < original_protein.index("*"):
        return "nonsense"
    elif original_protein != mutated_protein:
        return "missense"
    else:
        return "synonymous"

print(classify_mutation(hbb_cds_str, hbb_sickle))  # missense

### Exercise 4.2 — Synonymous and nonsense mutations (sample answers)

In [ ]:
# Synonymous: GAG (E) at codon 7 → GAA (E), position 20: G → A
hbb_syn = point_mutation(hbb_cds_str, 20, "A")
print("Codon 7 synonymous:", hbb_syn[18:21], "→", classify_mutation(hbb_cds_str, hbb_syn))

# Nonsense: GAG (E) at codon 7 → TAG (stop), position 18: G → T
hbb_nonsense = point_mutation(hbb_cds_str, 18, "T")
print("Codon 7 nonsense:  ", hbb_nonsense[18:21], "→", classify_mutation(hbb_cds_str, hbb_nonsense))

### Exercise 4.3 — Real HBB variants

Correct positions:
- **HbC** — p.Glu6Lys (mature), c.19G>A: position 18 (0-based), G→A, GAG→AAG (E→K)
- **HbE** — p.Glu26Lys (mature), c.79G>A: position 78 (0-based), G→A, GAG→AAG (E→K)
- **HbD** — p.Glu121Gln (mature), c.364G>C: position 363 (0-based), G→C, GAA→CAA (E→Q)

In [ ]:
# HbC: codon 7, position 18, G→A
hbb_HbC = point_mutation(hbb_cds_str, 18, "A")
print("HbC codon:", hbb_HbC[18:21], "→", classify_mutation(hbb_cds_str, hbb_HbC))
# HbC causes hemolytic anemia; heterozygotes have some protection against malaria

# HbE: codon 27, position 78, G→A
hbb_HbE = point_mutation(hbb_cds_str, 78, "A")
print("HbE codon:", hbb_HbE[78:81], "→", classify_mutation(hbb_cds_str, hbb_HbE))
# HbE is common in Southeast Asia; causes mild anemia; HbE/β-thal is a severe compound disease

# HbD: codon 122, position 363, G→C
hbb_HbD = point_mutation(hbb_cds_str, 363, "C")
print("HbD codon:", hbb_HbD[363:366], "→", classify_mutation(hbb_cds_str, hbb_HbD))
# HbD causes mild anemia; HbD/HbS compound causes sickle cell disease-like symptoms

---
## Section 5: Advanced Topics — Sample Solutions

### 5.1 Fetching from NCBI

In [ ]:
from Bio import Entrez, SeqIO

Entrez.email = "your.email@example.com"
handle = Entrez.efetch(db="nucleotide", id="NM_000518", rettype="gb", retmode="text")
record = SeqIO.read(handle, "genbank")
handle.close()

for feature in record.features:
    if feature.type == "CDS":
        fetched_cds = str(feature.extract(record.seq))
        break

print("Match:", fetched_cds == hbb_cds_str)

### 5.2 HGVS parser

In [ ]:
import re

def parse_hgvs(hgvs_string):
    match = re.match(r"c\.([0-9]+)([ACGT])>([ACGT])", hgvs_string)
    if not match:
        raise ValueError(f"Cannot parse: {hgvs_string}")
    pos_0based = int(match.group(1)) - 1
    return pos_0based, match.group(2), match.group(3)

# Sickle cell: c.20A>T
pos, ref, alt = parse_hgvs("c.20A>T")
print(f"Position (0-based): {pos}, {ref}→{alt}")
hbb_mut = point_mutation(hbb_cds_str, pos, alt)
print(classify_mutation(hbb_cds_str, hbb_mut))  # missense

### 5.3 Motif finding

In [ ]:
import re

def find_motif(sequence, motif):
    pattern = motif.replace("N", "[ACGT]").replace("W", "[AT]").replace("S", "[GC]")
    return [m.start() for m in re.finditer(f"(?={pattern})", str(sequence))]

# Use the region upstream of the CDS start in the full mRNA (fetch required)
# As a demo, search within hbb_cds_str itself
hits = find_motif(hbb_cds_str, "CCWWGG")
print(f"CCWWGG hits in HBB CDS: {hits}")